# 1. Model training and evaluation

    Prepare the four training configurations, run the existing DeepH-E3 training entry point, and evaluate the released four-model ensemble. The [configuration documentation](../models/config/README.md) records the shared settings and model-to-seed mapping. All evaluations use the same fixed ensemble; a common source dataset does not imply identical split membership across seeds and input orderings.

    Configuration preparation runs on a CPU. Full training and inference require a compatible DeepH-E3 installation, the preprocessed dataset and model checkpoints, and are disabled by default. Subsequent notebooks run small real-data analysis and visualization examples independently of training.

In [1]:
from pathlib import Path
import configparser
import importlib.util
import json
import os
import sys
import pandas as pd

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'tools/error_analysis.py').is_file() and (p / 'models/config').is_dir())
DOC = ROOT / 'doc'
OUT = DOC / 'output'
OUT.mkdir(exist_ok=True)
def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module
manifest = json.loads((ROOT / 'models/config/manifest.json').read_text())

## Prepare paths and working configurations

Download `Bilayer_graphene_dataset.zip` from [DeepH-E3 Dataset1](https://doi.org/10.5281/zenodo.7553640) and follow its README for preprocessing. Install [DeepH-E3](https://github.com/Xiaoxun-Gong/DeepH-E3), then integrate this repository's `kernel.py` and `parse_configs.py` as described in the repository README.

Set `DEEPHE3_ROOT`, `DEEPHE3_TRAIN_DATA`, `DEEPHE3_REVERSED_TRAIN_DATA` and `DEEPHE3_EVAL_DATA` to your local paths. The reversed-data path must contain the same structures in the intended reversed loader ordering. Merely changing the path name does not reverse a dataset. Preserve the original dataset index ordering if reproducing the archived splits exactly.

The cell creates working configurations under `doc/output`; it does not modify the released configurations. An evaluation configuration with `inference = False` requires DFT Hamiltonian labels and enables reference error analysis. For deployment without labels, use `inference = True` with the overlap data required by the upstream workflow.

In [2]:
DEEPHE3_ROOT = Path(os.environ.get('DEEPHE3_ROOT', str(ROOT / 'external/DeepH-E3')))
TRAIN_DATA = Path(os.environ.get('DEEPHE3_TRAIN_DATA', str(ROOT / 'data/Bilayer_graphene_dataset')))
REVERSED_DATA = Path(os.environ.get('DEEPHE3_REVERSED_TRAIN_DATA', str(ROOT / 'data/Bilayer_graphene_dataset_2')))
EVAL_DATA = Path(os.environ.get('DEEPHE3_EVAL_DATA', str(ROOT / 'data/evaluation_structures')))
working_configs = []
for index,item in enumerate(manifest):
    c = configparser.ConfigParser(interpolation=None)
    c.read(ROOT / item['copy'],encoding='utf-8')
    c['basic']['save_dir'] = str(OUT / item['model'] / 'training')
    c['data']['processed_data_dir'] = str(TRAIN_DATA if index < 2 else REVERSED_DATA)
    c['data']['save_graph_dir'] = str(OUT / item['model'] / 'graphs')
    c['data']['graph_dir'] = ''
    c['data']['DFT_data_dir'] = ''
    path = OUT / (item['model'] + '.ini')
    with path.open('w',encoding='utf-8') as handle: c.write(handle)
    working_configs.append(path)

checkpoint_dirs = [(ROOT / item['source']).parent.parent for item in manifest]
assert all((directory / 'best_model.pkl').is_file() for directory in checkpoint_dirs)
evaluation = configparser.ConfigParser(interpolation=None)
evaluation.read(ROOT / 'Bilayer_graphene_eval_ensemble.ini',encoding='utf-8')
evaluation['basic']['trained_model_dir'] = '||'.join(map(str,checkpoint_dirs))
evaluation['basic']['output_dir'] = str(OUT / 'ensemble_evaluation')
evaluation['data']['processed_data_dir'] = str(EVAL_DATA)
evaluation['data']['save_graph_dir'] = str(OUT / 'evaluation_graphs')
eval_ini = OUT / 'ensemble_evaluation.ini'
with eval_ini.open('w',encoding='utf-8') as handle: evaluation.write(handle)
print('Prepared four working train configurations and one ensemble evaluation configuration in doc/output.')

Prepared four working train configurations and one ensemble evaluation configuration in doc/output.


## Run training and inference in the DeepH-E3 environment

The next cell is disabled by default because full training is expensive and requires the external dataset and DeepH-E3 dependencies. Set the switches only when those inputs are ready. Training creates new checkpoints; the example inference configuration intentionally uses the released checkpoints. To evaluate retrained models, update `trained_model_dir` to their saved run directories.

The upstream command pattern is `python train.py CONFIG.ini` for training and `python evaluate.py CONFIG.ini` for evaluation. This notebook calls those existing entry points without introducing a different training or inference pipeline.

In [3]:
import subprocess
RUN_TRAINING = False
RUN_INFERENCE = False
if RUN_TRAINING:
    assert (DEEPHE3_ROOT / 'train.py').is_file()
    assert TRAIN_DATA.is_dir() and REVERSED_DATA.is_dir()
    for path in working_configs:
        subprocess.run([sys.executable,str(DEEPHE3_ROOT/'train.py'),str(path)],
                       cwd=DEEPHE3_ROOT,check=True)
if RUN_INFERENCE:
    assert (DEEPHE3_ROOT / 'evaluate.py').is_file() and EVAL_DATA.is_dir()
    subprocess.run([sys.executable,str(DEEPHE3_ROOT/'evaluate.py'),str(eval_ini)],
                   cwd=DEEPHE3_ROOT,check=True)
print('Full training requested:', RUN_TRAINING, '| Full inference requested:', RUN_INFERENCE)

Full training requested: False | Full inference requested: False


## Evaluation outputs

    With DFT labels and `inference = False`, the evaluator writes `test_result.h5` for reference-error analysis, ensemble-mean `hamiltonians_pred.h5`, and `hamiltonians_std.h5` for individual structures. The latter contains the population standard deviation across the four model predictions, rather than a variance.

    Continue with [data analysis and band calculations](02_data_analysis_and_bands.ipynb) to compute errors and bands, then [result visualization](03_result_visualization.ipynb) to generate individual figures. The included examples use stored evaluation outputs, so rerunning full training or inference is not required to execute those notebooks.